In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [5]:
question = 'How does the agentic loop keep calling the model until it stops?'

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

assistant = RAGBase(index, openai_client)

In [ ]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

(Response(id='resp_0705aee20b0f00e1006a36d957cdc4819eade525d32bc84060', created_at=1781979479.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0705aee20b0f00e1006a36d9583424819eb3308d6e98669119', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items.\n\n- If there was a function call, it runs the tool, appends the tool output to `messages`, and loops again.\n- If there were no function calls, it `break`s and stops.\n\nSo the stopping condition is: **no function calls in the latest model response**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781979480.0, conver

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [10]:
chunked_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

chunked_index.fit(chunks)

In [11]:
assistant2 = RAGBase(chunked_index, openai_client)
assistant2.rag('How does the agentic loop keep calling the model until it stops?')

(Response(id='resp_040b843af10e0073006a36daa4b904819281762510f6cde3d6', created_at=1781979812.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_040b843af10e0073006a36daa5612881928eb1a7bed69f109f', content=[ResponseOutputText(annotations=[], text='It keeps a `while True` loop and checks whether the model produced any `function_call` items in that turn.\n\n- If there is at least one function call, the code runs the tool, appends the tool result, and loops again.\n- If there are no function calls, it breaks out of the loop and returns the final answer.\n\nSo the stop condition is: **no function calls this turn means we’re done**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781979814.0, conversa

In [14]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [15]:
def search(query: str):
    """
    Search the course content for entries matching the given query.
    """
    return chunked_index.search(
        query,
        num_results=5,
    )

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

question = "How does the agentic loop work, and how is it different from plain RAG?"

In [16]:
# add tools
agent_tools = Tools()
agent_tools.add_tool(search)

In [17]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# define agent
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

result = runner.loop(
    prompt = question,
    callback = callback
)

-> Response received


-> Response received
